In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import date, timedelta
from pandas.tseries.offsets import * 
import calendar

pd.options.display.max_rows = 30

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 2, 11), datetime.date(2023, 2, 10))

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
to_mmdd = today.strftime('%m-%d')
print(f'Today in mm-dd format: {to_mmdd}')
print(f'yesterday: {yesterday}')

today: 2023-02-11
Today in mm-dd format: 02-11
yesterday: 2023-02-10 00:00:00


In [3]:
yesterday = yesterday.date()
one_month_ago = yesterday + DateOffset(days = -28)
print('One month ago: ', one_month_ago, ' ', one_month_ago.day_name())

One month ago:  2023-01-13 00:00:00   Friday


In [4]:
name = 'KCE'
new_grade = 'B3'
new_period = '4'

sqlUpd = """
UPDATE buy
SET grade = "%s", period = "%s" WHERE name = "%s"
"""
sqlUpd = sqlUpd % (new_grade, new_period, name)
print(sqlUpd)

rp = const.execute(sqlUpd)
rp.rowcount


UPDATE buy
SET grade = "B3", period = "4" WHERE name = "KCE"



1

In [ ]:
sql = '''
SELECT *
FROM buy
ORDER BY name'''
buy = pd.read_sql(sql, const)
buy['costAmt'] = buy.volbuy * buy.price
buy['date'] = pd.to_datetime(buy['date'])
buy.drop(['volsell', 'volbal'], axis=1, inplace = True)
buy.rename(columns={'volbuy':'shares'}, inplace = True)
buy['shares'] = buy['shares'].astype('int64')
buy.dtypes

In [ ]:
format_dict = {
    'shares':'{:,}',    
    'price':'{:.2f}',
    'costAmt':'{:,.2f}',    
    'dividend':'{:.4f}', 
    'date':'{:%Y-%m-%d}',    
}

In [ ]:
buy.style.format(format_dict)

In [ ]:
sql = '''
SELECT name, year, quarter, q_eps, aq_eps, publish_date
FROM epss
WHERE year = 2022 AND quarter = 2'''
epss = pd.read_sql(sql, conlt)
epss.sample(5)

In [ ]:
df_merge = pd.merge(buy, epss, on='name', how='inner')
df_merge.sort_values(['period','name'],ascending=[True,True])

In [ ]:
sql = '''
SELECT name, market
FROM stocks
ORDER BY name'''
my_stocks = pd.read_sql(sql, conmy)
my_stocks.shape

In [ ]:
filters = [
   (my_stocks.market.str.contains('SET50')),
   (my_stocks.market.str.contains('SET100')),
   (my_stocks.market.str.contains('mai'))    
]
values = ['SET50','SET100','mai']

In [ ]:
my_stocks["mrkt"] = np.select(filters, values, default='SET')
my_stocks.head()

In [ ]:
df_merge2 = pd.merge(df_merge,my_stocks,on='name',how='inner')
df_merge2

In [ ]:
cols = 'period grade mrkt name shares price costAmt '.split()
df_merge2['shares'] = df_merge['shares'].astype(int)
df_merge2[cols].sort_values(['period', 'name'],ascending=[True,True]).style.format(format_dict)

In [ ]:
df_merge2.groupby('period')['costAmt'].sum()

### End of Quarterly Process

In [ ]:
# look again at some of the data
catvars = ['period','grade']

for col in catvars:
  print(col, df_merge2[col].value_counts().sort_index(), sep="\n\n", end="\n\n\n")

In [ ]:
df_merge2.loc[
    df_merge2.grade == 'A2',
['period','name','grade','costAmt']].sort_values(['period','name'],ascending=[True,True]).style.format(format_dict)

In [ ]:
df_merge2.groupby('period').agg(
   {'costAmt':'sum','shares':'sum'}
).round(
   {'costAmt':2, 'shares':0}
).style.format(format_dict)

In [ ]:
df_merge2.replace(
   {
      'period':{'1':'Disposal','2':'Hi-Div','3':'Long Term','4':'Short Term'}
   }
).sample(5)

In [ ]:
cols = 'period grade name shares price costAmt dividend q_eps aq_eps mrkt'.split()
df_merge2.replace(
   {
      'period':{'1':'Disposal','2':'Hi-Div','3':'Long Term','4':'Short Term'}
   }
)[cols].sort_values(['period','name'],ascending=[True,True]).style.format(format_dict)

### Start of change grade after quarterly eps

In [ ]:
name = 'POPF'
sql = '''
SELECT * FROM buy
WHERE name = "%s"
'''
sql = sql % name
df = pd.read_sql(sql, con)
df

In [ ]:
sql = '''
UPDATE buy
SET period = "4", grade = "C1"
WHERE name = "%s"
'''
sql = sql % name
rp = con.execute(sql)
rp.rowcount

### End of change grade after quarterly eps

### Change period by name

In [ ]:
name = 'MAKRO'
period = '3'
sqlUpd = '''
UPDATE buy
SET period = "%s"
WHERE name =  "%s"'''
sqlUpd = sqlUpd % (period, name)
print(sqlUpd)

In [ ]:
rp = const.execute(sqlUpd)
rp.rowcount

### End of Change period according to grade

### Change grade of specific stock

In [ ]:
sql = """
UPDATE buy
SET period = "2"
WHERE name IN ('ANAN','LH','MCS','ORI')"""
print(sql)

In [ ]:
sql = """
UPDATE buy
SET period = "3"
WHERE name IN ('AMATA')"""
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

### End of Change grade of specific stock

In [ ]:
sql = """
INSERT INTO buy
VALUES('GC','2018-02-28',10000,5.3,)"""

In [ ]:
sql = '''
UPDATE buy
SET period = "4"
WHERE grade <>  "A4"'''
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
sql = '''
UPDATE buy
SET period = "1"
WHERE substr(grade,1,1) <>  "A"'''
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
rp = con.execute(sql)
rp.rowcount

### Individual record change

In [ ]:
sql = """
UPDATE buy
SET period = "2"
WHERE name IN ('GC','LH','TISCO','MCS','BR')"""
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

### Par value change

In [ ]:
sql = """
SELECT *
FROM price
LIMIT 1"""
prices = pd.read_sql(sql, con)
prices.dtypes

In [ ]:
name = "PTT"
effective_date = "2018-04-24"

In [ ]:
sql = """
UPDATE price
SET price = price*10, maxp = maxp*10, minp = minp*10, opnp = opnp*10,
qty = qty/10
WHERE name = '%s' AND date < '%s'
"""
sql = sql % (name, effective_date)
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
sql = """
SELECT *
FROM buy
LIMIT 1"""
buys = pd.read_sql(sql, con)
buys.dtypes

In [ ]:
sql = """
UPDATE buy
SET volbuy = 4000, price = 50.8
WHERE name = '%s' AND active = 1
"""
sql = sql % (name)
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

### Name change

In [ ]:
old_name = "EPCO"
new_name = "EP"

In [ ]:
sql = """
SELECT *
FROM stockname
WHERE name = '%s'
"""
sql = sql % (old_name)
stock = pd.read_sql(sql, con)
stock

In [ ]:
sql = """
UPDATE stockname
SET name = '%s'
WHERE name = '%s'
"""
sql = sql % (new_name, old_name)
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
sql = """
SELECT *
FROM price
WHERE name = '%s'
"""
sql = sql % (old_name)
price = pd.read_sql(sql, con)
price.shape

In [ ]:
sql = """
UPDATE price
SET name = '%s'
WHERE name = '%s'
"""
sql = sql % (new_name, old_name)
print(sql)

In [ ]:
rp = con.execute(sql)
rp.rowcount

In [ ]:
sql = '''
SELECT * FROM price WHERE name = 'ASK' ORDER BY date DESC'''
price = pd.read_sql(sql, con)
price

In [ ]:
sql = '''
DELETE FROM price WHERE date = "2021-12-14"'''
rp = con.execute(sql)
rp.rowcount

In [ ]:
sql = """
INSERT INTO buy (name, date, volbuy, price, volsell, volbal, active, dividend, period, grade)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
print(sql)

In [ ]:
rcd = ['TSE', '2022-01-21', 40000.0, 2.60, 0, 0, 1, 0.10, '3', 'A4']
rcd

In [ ]:
rp = con.execute(sql, rcd)
rp.rowcount

In [ ]:
sqlDel = '''
DELETE
FROM buy
WHERE name = "TSE"'''
rp = con.execute(sqlDel)
rp.rowcount

### SETIndex table

In [ ]:
sql = '''
SELECT * FROM setindex WHERE setindex IS Null'''
df = pd.read_sql(sql, con)
df

In [ ]:
sqlUpd = """
UPDATE setindex
SET setindex = 1661.89 WHERE setindex IS Null"""
print(sqlUpd)

In [ ]:
rp = con.execute(sqlUpd)
rp.rowcount

In [ ]:
sqlIns = """
INSERT INTO setindex
VALUES('2022-04-25',1675.33)"""
print(sqlIns)

In [ ]:
rp = con.execute(sqlIns)
rp.rowcount

### Temp process to solve problem

In [ ]:
sql = """
SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '%s' 
ORDER BY name, date"""
sql = sql % (today)
print(sql)

In [ ]:
df_price = pd.read_sql(sql, const)
df_price.shape

In [ ]:
sqlDel = """
DELETE
FROM price 
WHERE date = '%s' 
"""
sqlDel = sqlDel % (today)
print(sqlDel)

In [ ]:
rp = const.execute(sqlDel)
rp.rowcount

In [ ]:
sql = """
SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '%s' 
ORDER BY name, date"""
sql = sql % (yesterday)
print(sql)

In [ ]:
df_price_yesterday = pd.read_sql(sql, const)
df_price_yesterday.shape